# Teoría de Galois y Teoría de Grupos

Versión visual y algebraicamente más rica, apoyada en `sympy`, `numpy` y `matplotlib`, con animaciones integradas para explicar cómo las simetrías del grupo se traducen en simetrías de las raíces.

**Audiencia**
- Estudiantes de álgebra abstracta, teoría de números, física matemática y matemáticas puras.

**Prerrequisitos**
- Álgebra lineal básica.
- Polinomios, irreducibilidad y extensiones de cuerpos.
- Familiaridad con demostraciones y notación matemática formal.

**Objetivos**
- Describir grupos concretos con la API nativa de `sympy.combinatorics`.
- Entender por qué un grupo de Galois es un grupo de permutaciones de raíces con restricciones algebraicas.
- Visualizar las acciones de $\sigma$ y $\tau$ en $x^3 - 2$.
- Entender visualmente la correspondencia subgrupos $\leftrightarrow$ subcampos.
- Relacionar resolubilidad de grupos y resolubilidad por radicales.


## Tabla de Contenidos

1. Configuración simbólica y visual.
2. Grupos de permutaciones con `sympy`.
3. Animación de la acción de permutaciones en un triángulo.
4. Extensiones de cuerpos y cálculo exacto con `sympy`.
5. El grupo de Galois de $x^3 - 2$.
6. Animación de la acción sobre las raíces complejas.
7. Teorema Fundamental de la Teoría de Galois.
8. Animación de la correspondencia de Galois.
9. Resolubilidad: $S_3$ frente a $A_5$.
10. Ejercicios.

**Nota de uso**: las celdas animadas producen HTML/JavaScript embebido. En Jupyter pueden tardar un poco en renderizar la primera vez.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import sympy as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Math, display
from sympy.combinatorics import Permutation, PermutationGroup
from sympy.combinatorics.named_groups import AlternatingGroup

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

sp.init_printing()

# Paleta y objetos algebraicos centrales.
COLORS = {
    "amber": "#d97b29",
    "teal": "#0f8b8d",
    "ink": "#22303c",
    "rose": "#c8553d",
    "gold": "#f2cc8f",
    "slate": "#5c677d",
    "mist": "#eef2f5",
}

x = sp.symbols("x")
f = x**3 - 2
alpha = sp.real_root(2, 3)
omega = sp.exp(2 * sp.pi * sp.I / 3)

sigma = Permutation([1, 2, 0])  # (1 2 3)
tau = Permutation([0, 2, 1])    # (2 3)
G = PermutationGroup([sigma, tau])

root_symbols = [alpha, alpha * omega, alpha * omega**2]
root_labels = [r"$\alpha$", r"$\alpha\omega$", r"$\alpha\omega^2$"]
root_points = np.array([complex(sp.N(r, 30)) for r in root_symbols], dtype=np.complex128)


def print_table(headers, rows):
    widths = [len(str(h)) for h in headers]
    string_rows = []
    for row in rows:
        str_row = [str(item) for item in row]
        string_rows.append(str_row)
        for i, item in enumerate(str_row):
            widths[i] = max(widths[i], len(item))
    line = " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * w for w in widths)
    print(line)
    print(sep)
    for row in string_rows:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(headers))))


def perm_label(p):
    cycles = p.cyclic_form
    if not cycles:
        return "e"
    return "".join("(" + " ".join(str(i + 1) for i in cyc) + ")" for cyc in cycles)


def group_elements(group):
    return sorted(
        list(group.generate_schreier_sims()),
        key=lambda g: tuple(g.array_form[: group.degree]),
    )


def parity_label(p):
    return "par" if p.signature() == 1 else "impar"


def smooth_step(t):
    return 0.5 - 0.5 * np.cos(np.pi * t)


def display_animation(anim):
    html = HTML(anim.to_jshtml(default_mode="loop"))
    plt.close(anim._fig)
    display(html)


print("Entorno listo con numpy, sympy, matplotlib e IPython.")


## 1. Grupos de permutaciones con `sympy`

La teoría de Galois entra naturalmente en la categoría de grupos de permutaciones. Si un polinomio separable tiene $n$ raíces, cualquier automorfismo del cuerpo de descomposición que fije el cuerpo base debe enviar raíces en raíces, y por tanto define una permutación de un conjunto de tamaño $n$.

Trabajaremos con
\[
\sigma = (1\,2\,3), \qquad \tau = (2\,3),
\]
que generan un grupo de orden 6 isomorfo a $S_3$.

La ventaja de `sympy.combinatorics` es que permite operar con estos objetos de forma nativa: orden, signo, series derivadas y generación de grupos.


In [ ]:
elements = group_elements(G)
rows = []
for g in elements:
    rows.append([perm_label(g), g.order(), parity_label(g), g.array_form[: G.degree]])

print("Resumen de los elementos de G = <sigma, tau>:")
print_table(["Permutacion", "Orden", "Paridad", "Forma arreglo"], rows)
print()
print("|G| =", G.order())
print("sigma^3 = e ?", sigma**3 == Permutation(list(range(3))))
print("tau^2 = e ?", tau**2 == Permutation(list(range(3))))
print("tau * sigma * tau = sigma^{-1} ?", tau * sigma * tau == sigma**-1)
display(Math(r"G = \langle \sigma, \tau \mid \sigma^3 = \tau^2 = e,\ \tau\sigma\tau = \sigma^{-1} \rangle \cong S_3"))


## 2. Animación: la acción de una permutación sobre un triángulo etiquetado

Esta primera animación no usa aún raíces; solo muestra la idea abstracta. Cada vértice del triángulo representa una etiqueta. Cuando aplicamos una permutación, la etiqueta viaja a su nueva posición.

Observa dos patrones distintos:
- $\sigma$ produce una rotación cíclica de las tres etiquetas.
- $\tau$ fija la primera etiqueta e intercambia las otras dos.

Esa dicotomía "ciclo de orden 3 + transposición" es exactamente la estructura que reaparecerá en el grupo de Galois de $x^3 - 2$.


In [ ]:
base_angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, 4)[:-1]
base_positions = np.column_stack((np.cos(base_angles), np.sin(base_angles)))


def animate_triangle_action(permutation, title):
    target_positions = base_positions[np.array(permutation.array_form[:3])]
    fig, ax = plt.subplots(figsize=(5.8, 5.8), facecolor="#fbfaf7")
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.35, 1.45)
    ax.set_aspect("equal")
    ax.axis("off")

    for i, j in [(0, 1), (1, 2), (2, 0)]:
        ax.plot(base_positions[[i, j], 0], base_positions[[i, j], 1], color="#d8dee6", lw=1.5, zorder=1)
        ax.plot(target_positions[[i, j], 0], target_positions[[i, j], 1], color="#f0c9a4", lw=1.0, ls="--", zorder=1)

    colors = [COLORS["amber"], COLORS["teal"], COLORS["rose"]]
    scatter = ax.scatter(base_positions[:, 0], base_positions[:, 1], s=520, c=colors, ec="white", lw=2, zorder=3)
    ghost = ax.scatter(target_positions[:, 0], target_positions[:, 1], s=520, c=colors, alpha=0.12, zorder=2)
    texts = [
        ax.text(*(base_positions[i] + np.array([0.0, 0.13])), f"{i+1}", ha="center", va="center", fontsize=16, weight="bold", color=COLORS["ink"])
        for i in range(3)
    ]
    ax.text(0, 1.24, title, ha="center", va="center", fontsize=16, weight="bold", color=COLORS["ink"])
    ax.text(0, -1.18, "Líneas punteadas: posiciones destino de las etiquetas", ha="center", fontsize=11, color=COLORS["slate"])

    def update(frame):
        t = smooth_step(frame / 40)
        positions = (1.0 - t) * base_positions + t * target_positions
        scatter.set_offsets(positions)
        for i, txt in enumerate(texts):
            txt.set_position(positions[i] + np.array([0.0, 0.13]))
        return [scatter, ghost, *texts]

    anim = FuncAnimation(fig, update, frames=41, interval=55, blit=True)
    display_animation(anim)


animate_triangle_action(sigma, r"Acción de $\sigma = (1\,2\,3)$")
animate_triangle_action(tau, r"Acción de $\tau = (2\,3)$")


## 3. Extensiones de cuerpos y cálculo exacto con `sympy`

Consideremos el polinomio
\[
f(x)=x^3-2 \in \mathbb{Q}[x].
\]

Hechos algebraicos esenciales:

1. Es irreducible sobre $\mathbb{Q}$ por Eisenstein con el primo $2$.
2. Una raíz real es $\alpha = \sqrt[3]{2}$.
3. Si $\omega = e^{2\pi i/3}$ es una raíz cúbica primitiva de la unidad, entonces las tres raíces son
   \[
   \alpha,\quad \alpha\omega,\quad \alpha\omega^2.
   \]
4. El cuerpo de descomposición es
   \[
   K=\mathbb{Q}(\alpha,\omega).
   \]

`sympy` nos permite mantener las expresiones exactas y, al mismo tiempo, obtener aproximaciones numéricas cuando queremos visualizarlas.


In [ ]:
exact_roots = [sp.simplify(r) for r in root_symbols]
numeric_roots = [complex(z) for z in sp.nroots(f, n=20)]
disc = sp.discriminant(f, x)
phi3 = sp.minimal_polynomial(omega, x)

display(Math(rf"f(x) = {sp.latex(f)}"))
display(Math(rf"\operatorname{{disc}}(f) = {sp.latex(disc)}"))
display(Math(rf"\Phi_3(x) = \operatorname{{minpoly}}(\omega) = {sp.latex(phi3)}"))

rows = []
for idx, (r_exact, r_num) in enumerate(zip(exact_roots, numeric_roots), start=1):
    rows.append([f"r_{idx}", sp.latex(r_exact), f"{r_num.real:.6f} {r_num.imag:+.6f}i"])

print("Raíces exactas y aproximadas de x^3 - 2:")
print_table(["Raiz", "Forma exacta", "Aproximacion numerica"], rows)
print()
print("Interpretacion:")
print("- El discriminante -108 es no cuadrado en Q, lo que ya sugiere la presencia de una permutacion impar en el grupo de Galois.")
print("- [Q(alpha):Q] = 3 y [Q(omega):Q] = 2; en este caso el grado total del cuerpo de descomposicion es 6.")


## 4. El grupo de Galois de $x^3 - 2$

Los automorfismos de $K=\mathbb{Q}(\alpha,\omega)$ que fijan $\mathbb{Q}$ deben respetar simultáneamente:
\[
\alpha^3 = 2, \qquad \omega^2 + \omega + 1 = 0.
\]

Por eso,
\[
\sigma(\alpha)=\alpha\omega,\qquad \sigma(\omega)=\omega,
\]
\[
\tau(\alpha)=\alpha,\qquad \tau(\omega)=\omega^2
\]
son automorfismos legítimos. En la acción sobre las raíces:

- $\sigma$ induce el 3-ciclo $(1\,2\,3)$,
- $\tau$ induce la transposición $(2\,3)$.

Como un 3-ciclo y una transposición generan $S_3$, obtenemos
\[
\operatorname{Gal}(K/\mathbb{Q}) \cong S_3.
\]


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6.1), facecolor="#fbfaf7")
ax.scatter(root_points.real, root_points.imag, s=260, c=[COLORS["amber"], COLORS["teal"], COLORS["rose"]], ec="white", lw=1.8, zorder=3)

for label, z, color in zip(root_labels, root_points, [COLORS["amber"], COLORS["teal"], COLORS["rose"]]):
    ax.text(z.real + 0.07, z.imag + 0.07, label, fontsize=14, color=color, weight="bold")

polygon = np.r_[root_points, root_points[0]]
ax.plot(polygon.real, polygon.imag, color=COLORS["slate"], lw=1.6, alpha=0.75)
ax.axhline(0, color="#cfd8df", lw=1)
ax.axvline(0, color="#cfd8df", lw=1)
ax.set_title(r"Raíces de $x^3 - 2$ en el plano complejo", fontsize=15, color=COLORS["ink"])
ax.set_xlabel("Parte real")
ax.set_ylabel("Parte imaginaria")
ax.set_aspect("equal")
plt.show()

print("sigma induce:", perm_label(sigma))
print("tau induce:", perm_label(tau))
print("Orden del grupo generado:", G.order())


## 5. Animación: cómo actúan $\sigma$ y $\tau$ sobre las raíces

Esta es la animación central del notebook. El triángulo ahora no representa etiquetas abstractas, sino las tres raíces complejas de $x^3-2$.

Lee la escena así:
- cada color representa una raíz concreta;
- el punto se mueve hacia la imagen de esa raíz bajo el automorfismo;
- la estructura geométrica del conjunto de raíces permanece, pero las etiquetas se reorganizan.

Esa reorganización es exactamente la acción del grupo de Galois.


In [ ]:
def animate_root_action(permutation, title, subtitle):
    start = root_points
    target = root_points[np.array(permutation.array_form[:3])]
    fig, ax = plt.subplots(figsize=(6.6, 6.2), facecolor="#fbfaf7")
    ax.set_aspect("equal")
    ax.set_xlim(-1.65, 1.65)
    ax.set_ylim(-1.45, 1.45)
    ax.axhline(0, color="#d8dee6", lw=1)
    ax.axvline(0, color="#d8dee6", lw=1)
    ax.grid(alpha=0.18)

    ax.scatter(target.real, target.imag, s=360, c=[COLORS["amber"], COLORS["teal"], COLORS["rose"]], alpha=0.10, zorder=1)
    ax.plot(np.r_[start.real, start.real[0]], np.r_[start.imag, start.imag[0]], ls="--", lw=1.2, color="#d8dee6")

    scatter = ax.scatter(start.real, start.imag, s=280, c=[COLORS["amber"], COLORS["teal"], COLORS["rose"]], ec="white", lw=2, zorder=3)
    texts = [
        ax.text(start[i].real + 0.05, start[i].imag + 0.06, root_labels[i], fontsize=14, color=[COLORS["amber"], COLORS["teal"], COLORS["rose"]][i], weight="bold")
        for i in range(3)
    ]
    ax.set_title(title, fontsize=15, color=COLORS["ink"])
    ax.text(0.0, -1.33, subtitle, ha="center", fontsize=11, color=COLORS["slate"])

    def update(frame):
        t = smooth_step(frame / 45)
        positions = (1.0 - t) * start + t * target
        scatter.set_offsets(np.column_stack((positions.real, positions.imag)))
        for i, txt in enumerate(texts):
            txt.set_position((positions[i].real + 0.05, positions[i].imag + 0.06))
        return [scatter, *texts]

    anim = FuncAnimation(fig, update, frames=46, interval=55, blit=True)
    display_animation(anim)


animate_root_action(
    sigma,
    r"Acción de $\sigma$: $\alpha \mapsto \alpha\omega$",
    r"El automorfismo recorre cíclicamente las tres raíces: $r_1 \to r_2 \to r_3 \to r_1$.",
)
animate_root_action(
    tau,
    r"Acción de $\tau$: conjugación compleja",
    r"$\tau$ fija la raíz real y refleja las dos raíces complejas conjugadas.",
)


## 6. Teorema Fundamental de la Teoría de Galois

Si $L/K$ es una extensión finita galoisiana y $G=\operatorname{Gal}(L/K)$, existe una correspondencia biyectiva e inclusiones inversas entre:

- subgrupos $H \le G$,
- subcampos intermedios $K \subseteq E \subseteq L$.

La asignación es
\[
H \longmapsto L^H = \{x \in L : \sigma(x)=x \ \forall \sigma \in H\}.
\]

Para el caso $K=\mathbb{Q}(\alpha,\omega)$ sobre $\mathbb{Q}$:

- el grupo total fija exactamente $\mathbb{Q}$;
- el subgrupo cíclico de orden 3 fija $\mathbb{Q}(\omega)$;
- cada subgrupo de orden 2 fija una cúbica no normal;
- el subgrupo trivial fija todo el cuerpo de descomposición.


In [ ]:
correspondence_rows = [
    [r"$G = S_3$", r"$\mathbb{Q}$", "Todo automorfismo fija solo el cuerpo base"],
    [r"$A_3 = \langle \sigma \rangle$", r"$\mathbb{Q}(\omega)$", "Subgrupo normal; extensión cuadrática normal"],
    [r"$\langle \tau \rangle$", r"$\mathbb{Q}(\sqrt[3]{2})$", "Subgrupo no normal; cúbica no normal"],
    [r"$\langle \sigma\tau \rangle$", r"$\mathbb{Q}(\sqrt[3]{2}\,\omega)$", "Conjugado del anterior"],
    [r"$\langle \sigma^2\tau \rangle$", r"$\mathbb{Q}(\sqrt[3]{2}\,\omega^2)$", "Conjugado del anterior"],
    [r"$\{e\}$", r"$\mathbb{Q}(\sqrt[3]{2}, \omega)$", "Subgrupo trivial; fija todo K"],
]

print_table(["Subgrupo", "Campo fijo", "Comentario"], correspondence_rows)


## 7. Animación: la correspondencia de Galois como un puente entre dos mundos

La siguiente animación organiza la información en dos columnas:

- a la izquierda, subgrupos del grupo de Galois;
- a la derecha, subcampos fijos correspondientes.

Lo importante no es solo la biyección, sino la inversión del orden:
subgrupos más grandes corresponden a campos fijos más pequeños.


In [ ]:
subgroup_labels = [
    r"$S_3$",
    r"$A_3$",
    r"$\langle \tau \rangle$",
    r"$\langle \sigma\tau \rangle$",
    r"$\langle \sigma^2\tau \rangle$",
    r"$\{e\}$",
]
field_labels = [
    r"$\mathbb{Q}$",
    r"$\mathbb{Q}(\omega)$",
    r"$\mathbb{Q}(\sqrt[3]{2})$",
    r"$\mathbb{Q}(\sqrt[3]{2}\,\omega)$",
    r"$\mathbb{Q}(\sqrt[3]{2}\,\omega^2)$",
    r"$K = \mathbb{Q}(\sqrt[3]{2},\omega)$",
]
comments = [
    "Grupo completo ↔ cuerpo base",
    "Subgrupo normal de orden 3 ↔ subextensión normal cuadrática",
    "Subgrupo de orden 2 ↔ cúbica no normal",
    "Subgrupo conjugado ↔ campo conjugado",
    "Otro subgrupo conjugado ↔ otro campo conjugado",
    "Subgrupo trivial ↔ todo el cuerpo de descomposición",
]

left_y = np.linspace(5, 0, 6)
right_y = np.linspace(5, 0, 6)


def animate_galois_correspondence():
    fig, ax = plt.subplots(figsize=(11, 6.5), facecolor="#fbfaf7")
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.8, 5.8)
    ax.axis("off")

    ax.text(0.20, 5.45, "Subgrupos de Gal(K/Q)", ha="center", fontsize=16, weight="bold", color=COLORS["ink"])
    ax.text(0.80, 5.45, "Subcampos fijos", ha="center", fontsize=16, weight="bold", color=COLORS["ink"])

    lines = []
    left_texts = []
    right_texts = []

    for i, (ys, yf) in enumerate(zip(left_y, right_y)):
        line, = ax.plot([0.30, 0.70], [ys, yf], color="#d8dee6", lw=1.4, alpha=0.7, zorder=1)
        lines.append(line)
        lt = ax.text(
            0.18, ys, subgroup_labels[i], ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.35", fc=COLORS["mist"], ec="none"), color=COLORS["ink"]
        )
        rt = ax.text(
            0.82, yf, field_labels[i], ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.35", fc=COLORS["mist"], ec="none"), color=COLORS["ink"]
        )
        left_texts.append(lt)
        right_texts.append(rt)

    footer = ax.text(0.5, -0.35, "", ha="center", va="center", fontsize=12, color=COLORS["slate"])
    order_note = ax.text(0.5, -0.60, "La inclusión se invierte: H1 ⊆ H2 implica K^{H2} ⊆ K^{H1}.", ha="center", fontsize=11, color=COLORS["slate"])

    def update(frame):
        idx = (frame // 18) % len(lines)
        for j in range(len(lines)):
            active = (j == idx)
            lines[j].set_color(COLORS["teal"] if active else "#d8dee6")
            lines[j].set_linewidth(3.4 if active else 1.4)
            lines[j].set_alpha(1.0 if active else 0.45)
            left_texts[j].set_bbox(dict(boxstyle="round,pad=0.35", fc=(COLORS["gold"] if active else COLORS["mist"]), ec="none"))
            right_texts[j].set_bbox(dict(boxstyle="round,pad=0.35", fc=(COLORS["gold"] if active else COLORS["mist"]), ec="none"))
        footer.set_text(comments[idx])
        return [*lines, *left_texts, *right_texts, footer, order_note]

    anim = FuncAnimation(fig, update, frames=18 * len(lines), interval=120, blit=True)
    display_animation(anim)


animate_galois_correspondence()


## 8. Resolubilidad: contraste entre $S_3$ y $A_5$

La serie derivada de un grupo
\[
G \trianglerighteq [G,G] \trianglerighteq [[G,G],[G,G]] \trianglerighteq \cdots
\]
mide cuán lejos está el grupo de ser abeliano. Un grupo es **resoluble** si esa serie llega al grupo trivial en un número finito de pasos.

La conexión de Galois con las ecuaciones polinómicas dice:

> Un polinomio separable en característica 0 es resoluble por radicales si su grupo de Galois es resoluble.

Esto explica por qué la cúbica general sí admite fórmula por radicales, mientras que el quintico general no: el grupo alternante $A_5$ no es resoluble.


In [ ]:
S3_series = G.derived_series()
A5 = AlternatingGroup(5)
A5_series = A5.derived_series()

rows = [
    ["S3", [H.order() for H in S3_series], bool(G.is_solvable)],
    ["A5", [H.order() for H in A5_series[:3]], bool(A5.is_solvable)],
]

print_table(["Grupo", "Ordenes en la serie derivada", "Resoluble"], rows)

fig, ax = plt.subplots(figsize=(7.2, 4.2), facecolor="#fbfaf7")
s3_sizes = [H.order() for H in S3_series]
a5_sizes = [H.order() for H in A5_series[:3]]
ax.plot(range(len(s3_sizes)), s3_sizes, marker="o", lw=2.5, color=COLORS["teal"], label=r"$S_3$")
ax.plot(range(len(a5_sizes)), a5_sizes, marker="s", lw=2.5, color=COLORS["rose"], label=r"$A_5$")
ax.set_xticks(range(max(len(s3_sizes), len(a5_sizes))))
ax.set_xlabel("Paso en la serie derivada")
ax.set_ylabel("Orden del subgrupo")
ax.set_title("Serie derivada: colapso de S3 frente a estabilidad de A5", fontsize=14, color=COLORS["ink"])
ax.legend()
plt.show()


## 9. Ejercicios

1. Modifica la animación del triángulo para representar la permutación $(1\,3)$ y describe geométricamente su acción.
2. Usa `sympy` para construir explícitamente el grupo alternante $A_4$ y verifica si es resoluble.
3. Repite la visualización de raíces para $x^4 - 2$ y explica por qué aparece una simetría compatible con un subgrupo de $D_4$.
4. En el ejemplo de $x^3 - 2$, explica por qué $\mathbb{Q}(\sqrt[3]{2})/\mathbb{Q}$ no es normal aunque provenga de una subextensión del cuerpo de descomposición.
5. Demuestra que los subgrupos conjugados de orden 2 de $S_3$ corresponden a subcampos conjugados.


In [ ]:
# Celda guia para experimentar.

transposition_13 = Permutation([2, 1, 0])
print("Nueva permutacion:", perm_label(transposition_13))
print("Orden:", transposition_13.order())
print("Paridad:", parity_label(transposition_13))

A4 = PermutationGroup([
    Permutation([1, 2, 0, 3]),  # (1 2 3)
    Permutation([1, 3, 2, 0]),  # (1 2 4)
])
print("Orden de A4 construido explicitamente:", A4.order())
print("A4 es resoluble?:", bool(A4.is_solvable))

# Puedes reutilizar animate_triangle_action sobre una permutacion de 3 elementos,
# o adaptar animate_root_action a otro polinomio cambiando f, root_symbols y root_points.


## Referencias

- Artin, M. *Algebra*.
- Dummit, D. y Foote, R. *Abstract Algebra*.
- Stewart, I. *Galois Theory*.
- Cox, D. *Galois Theory*.
- Lang, S. *Algebra*.

**Comentario final**:
la teoría de Galois se vuelve mucho más transparente cuando se piensa como una teoría de simetrías actuando sobre configuraciones algebraicas. Las animaciones no sustituyen a la demostración, pero ayudan a ver la estructura antes de formalizarla.
